In [ ]:
# ============================================================
# LAB 3.5 - AA3 No.1
# Voice Cloning dengan Qwen3-TTS (model ID diperbarui)
# Model: Qwen/Qwen3-TTS-12Hz-0.6B-Base
# Library: qwen-tts (bukan transformers)
# ============================================================

# ============================================================
# STEP 1 - Install
# ============================================================
!pip install qwen-tts soundfile torchaudio

In [ ]:
# ============================================================
# STEP 2 - Load Model
# ============================================================
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

MODEL_NAME = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print("Loading Qwen3-TTS model... (~2.5GB, tunggu sebentar)")

model = Qwen3TTSModel.from_pretrained(
    MODEL_NAME,
    device_map=DEVICE,
    dtype=torch.bfloat16,
)
print("Model loaded!")

In [ ]:
# ============================================================
# STEP 3 - Rekam Suara 10 Detik
# ============================================================
from IPython.display import Javascript, display, Audio
from google.colab import output
import base64, io
import time

RECORD_JS = """
const sleep = (ms) => new Promise(resolve => setTimeout(resolve, ms));
const b64 = (buf) => {
  var binary = '';
  var bytes = new Uint8Array(buf);
  for (var i = 0; i < bytes.byteLength; i++)
    binary += String.fromCharCode(bytes[i]);
  return window.btoa(binary);
};
async function recordAudio(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(seconds * 1000);
  recorder.stop();
  await sleep(500);
  const blob = new Blob(chunks);
  const buf = await blob.arrayBuffer();
  google.colab.kernel.invokeFunction('notebook.set_audio', [b64(buf)], {});
}
recordAudio(10);
"""

audio_data = {}

def set_audio(b64_data):
    audio_data['bytes'] = base64.b64decode(b64_data)


output.register_callback('notebook.set_audio', set_audio)
print("Rekam sekarang! Bicara selama 10 detik dengan jelas...")
display(Javascript(RECORD_JS))
time.sleep(10)
print("Rekam selesai!")

In [ ]:
print(audio_data.keys())
print(audio_data)

In [ ]:
RECORD_JS_3 = """
const sleep = (ms) => new Promise(resolve => setTimeout(resolve, ms));
const b64 = (buf) => {
  var binary = '';
  var bytes = new Uint8Array(buf);
  for (var i = 0; i < bytes.byteLength; i++)
    binary += String.fromCharCode(bytes[i]);
  return window.btoa(binary);
};
async function recordAudio(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(seconds * 1000);
  recorder.stop();
  await sleep(500);
  const blob = new Blob(chunks);
  const buf = await blob.arrayBuffer();
  google.colab.kernel.invokeFunction('notebook.set_audio', [b64(buf)], {});
}
recordAudio(3);
"""
audio_data_3 = {}

def set_audio_3(b64_data):
    audio_data_3['bytes'] = base64.b64decode(b64_data)

output.register_callback('notebook.set_audio', set_audio_3)
print("Rekam sekarang! Bicara selama 3 detik dengan jelas...")
display(Javascript(RECORD_JS_3))
time.sleep(3)
print("Rekam selesai!")

In [ ]:
print(audio_data_3.keys())
print(audio_data_3)

In [ ]:
print(audio_data.keys())
print(audio_data)

In [ ]:
with open("recording.webm", "wb") as f:
    f.write(audio_data["bytes"])

with open("recording_3.webm", "wb") as f:
    f.write(audio_data_3["bytes"])

print("Recording berhasil disimpan.")

In [ ]:
!apt-get -qq install ffmpeg

In [ ]:
!ffmpeg -i recording.webm my_reference.wav -y
!ffmpeg -i recording_3.webm my_reference_3.wav -y

In [ ]:
from IPython.display import Audio

Audio("my_reference.wav")


In [ ]:
Audio("my_reference_3.wav")

In [ ]:
# ============================================================
# STEP 5 - Clone Suara
# PENTING: ref_text diisi dengan teks yang kamu ucapkan
#          di rekaman referensi (step 3)
# ============================================================
REF_TEXT = "Nama saya Muhammad Rafid Kresnanda, mahasiswa teknik informatika di universitas muhammadiyah surabaya"
TEXT     = "Nama saya Muhammad Rafid Kresnanda, mahasiswa teknik informatika di umsura"

print("\nProses voice cloning...")
wavs, sr = model.generate_voice_clone(
    text=TEXT,
    language="auto",
    ref_audio="my_reference.wav",
    ref_text=REF_TEXT,
)

sf.write("my_cloned_voice.wav", wavs[0], sr)
print("Cloned voice saved to: my_cloned_voice.wav")

# Dengarkan hasilnya
print("\n--- Suara Asli ---")
display(Audio("my_reference.wav"))
print("--- Hasil Clone ---")
display(Audio("my_cloned_voice.wav"))